# Item-Based 추천 시스템을 위한 전처리 노트북

이 노트북에서는 **MovieLens / The Movies Dataset 원본 데이터**를 기반으로,
Item-Based 협업 필터링에 활용하기 위한 전처리를 단계별로 수행한다.

데이터 원본 파일:
- `movies_metadata.csv` : 영화 메타데이터 (장르, 개봉일, 수익 등)
- `ratings.csv` : 사용자–영화 평점 데이터
- `tags.csv` : 사용자 태그 데이터

진행 순서 (A → B → C):
1. **A. `movies_metadata` 정제** – 영화 아이템 feature 품질 개선
2. **B. `ratings` 정규화** – Item-Based 평점 기반 유사도 계산을 위한 전처리
3. **C. `tags` 전처리** – 태그 기반 콘텐츠 feature 구축 (TF-IDF 등) 전 단계


In [3]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path

# =====================================================================
# 공통 설정
# =====================================================================
# 실제 사용할 때는 아래 DATA_DIR를 자신의 폴더 구조에 맞게 수정하면 된다.
# (이 노트북에서는 같은 폴더에 CSV 파일이 있다고 가정한다.)
DATA_DIR = Path('.')  # 예: Path('data')

MOVIES_PATH = DATA_DIR / 'movies_metadata.csv'
RATINGS_PATH = DATA_DIR / 'ratings.csv'
TAGS_PATH = DATA_DIR / 'tags.csv'

OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Movies path :', MOVIES_PATH.resolve())
print('Ratings path:', RATINGS_PATH.resolve())
print('Tags path   :', TAGS_PATH.resolve())
print('Output dir  :', OUTPUT_DIR.resolve())


Movies path : C:\Users\gram\Downloads\item_based-recommendation-system\movies_metadata.csv
Ratings path: C:\Users\gram\Downloads\item_based-recommendation-system\ratings.csv
Tags path   : C:\Users\gram\Downloads\item_based-recommendation-system\tags.csv
Output dir  : C:\Users\gram\Downloads\item_based-recommendation-system\processed


## A. `movies_metadata` 정제

Item-Based 추천에서 **아이템(영화) 간 유사도**를 계산할 때,
장르, 개봉 연도, 평점 수, 인기도(popularity) 등 메타데이터를 feature로 사용할 수 있다.
따라서 다음과 같은 정제가 필요하다.

- 잘못된 타입(`id`가 숫자가 아닌 경우 등) 제거
- JSON 문자열 형태의 `genres` 파싱 → 장르 리스트 feature
- 날짜/숫자 컬럼 형 변환 (`release_date`, `budget`, `revenue`, `runtime`, `vote_average`, `vote_count`, `popularity`)
- 명시적으로 필요 없는 컬럼(전부 NaN인 `Unnamed:` 컬럼 등) 제거

※ MovieLens `ratings.csv`의 `movieId`와 직접 매핑하기 위해서는
**`links.csv` (movieId ↔ tmdbId 매핑)**가 추가로 필요하다.
이 노트북에서는 **콘텐츠 feature 정제까지** 수행하고,
MovieLens와의 매핑은 추후 단계에서 별도로 수행한다고 가정한다.


In [4]:
# =====================================================================
# A-1. movies_metadata 로딩
# =====================================================================
# 원본 파일은 종종 UTF-8이 아니라 latin1 인코딩으로 되어 있으므로,
# UnicodeDecodeError 방지를 위해 encoding='latin1'를 명시한다.
movies_raw = pd.read_csv(MOVIES_PATH, low_memory=False, encoding='latin1')
print('원본 movies_metadata shape:', movies_raw.shape)
print('컬럼 예시:', movies_raw.columns.tolist()[:15])
movies_raw.head(3)


원본 movies_metadata shape: (45442, 45)
컬럼 예시: ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date']


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44
0,FALSE,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FALSE,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FALSE,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# =====================================================================
# A-2. 의미 없는 Unnamed 컬럼 제거
# =====================================================================
# CSV에서 잘못 저장된 경우, 'Unnamed: xx'와 같이 전부 NaN인 컬럼이 생기는 경우가 많다.
# 이런 컬럼은 Item-Based feature 계산에 아무 의미가 없으므로 제거한다.
unnamed_cols = [c for c in movies_raw.columns if c.startswith('Unnamed')]
print('제거할 Unnamed 컬럼들:', unnamed_cols)

movies = movies_raw.drop(columns=unnamed_cols)
print('Unnamed 제거 후 shape:', movies.shape)


제거할 Unnamed 컬럼들: ['Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44']
Unnamed 제거 후 shape: (45442, 24)


In [6]:
# =====================================================================
# A-3. id 컬럼 정제 (tmdbId로 사용)
# =====================================================================
# movies_metadata의 'id' 컬럼은 TMDB 영화 ID인데,
# 일부 row는 숫자가 아닌 값(예: '1997-08-20', 'test')이 섞여 있다.
# 숫자로 변환 가능한 row만 남기고 나머지는 제거한다.
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
before_drop = len(movies)
movies = movies.dropna(subset=['id'])
after_drop = len(movies)

print(f"'id' 숫자 변환 불가로 인해 제거된 행 수: {before_drop - after_drop}")
movies['id'] = movies['id'].astype(int)

# 명확한 의미를 위해 'tmdbId'로 컬럼명을 바꾸어둔다.
movies = movies.rename(columns={'id': 'tmdbId'})


'id' 숫자 변환 불가로 인해 제거된 행 수: 7


In [7]:
# =====================================================================
# A-4. genres JSON 문자열 파싱
# =====================================================================
# 'genres' 컬럼은 문자열 형태의 리스트 예시:
#   "[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}]"
# 이런 문자열을 파싱하여 장르 이름 리스트로 변환한다.
# Item-Based 콘텐츠 유사도에서 주요 feature가 될 수 있다.
def parse_genres(x):
    if pd.isna(x) or x == '[]':
        return []
    try:
        data = ast.literal_eval(x)
        if isinstance(data, list):
            return [d.get('name') for d in data if isinstance(d, dict) and 'name' in d]
    except Exception:
        pass
    return []

movies['genres_list'] = movies['genres'].apply(parse_genres)

# 대표 장르(첫 번째 장르)를 간단히 지정해 두면,
# EDA나 간단한 분석에서 편리하게 사용할 수 있다.
movies['main_genre'] = movies['genres_list'].apply(lambda lst: lst[0] if lst else None)

movies[['title', 'genres', 'genres_list', 'main_genre']].head(5)


,title,genres,genres_list,main_genre
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[Animation, Comedy, Family]",Animation
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[Adventure, Fantasy, Family]",Adventure
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[Romance, Comedy]",Romance
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","[Comedy, Drama, Romance]",Comedy
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",[Comedy],Comedy


In [8]:
# =====================================================================
# A-5. 날짜 및 숫자형 feature 정리
# =====================================================================
# Item-Based에서 사용 가능한 대표적인 수치형 feature들:
# - release_year : 영화 시대/세대 구분
# - runtime      : 상영 시간
# - vote_average : 평점(0~10)
# - vote_count   : 평점을 남긴 사람 수 (신뢰도 가중치에 활용 가능)
# - popularity   : TMDB에서 제공하는 인기도 지표
# - budget, revenue : 예산/수익(필요 시 로그 스케일 사용)
#
# 이 컬럼들을 숫자/날짜형으로 변환하고, 간단한 후처리를 해둔다.
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['release_year'] = movies['release_date'].dt.year

num_cols = ['budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity']
for c in num_cols:
    movies[c] = pd.to_numeric(movies[c], errors='coerce')

# budget, revenue는 0 또는 이상치가 많아서 그대로 쓰기보다는
# 로그 스케일을 사용하는 경우가 많다.
# (0이거나 결측인 경우는 0으로 두고, 나머지만 log1p를 적용한다.)
for c in ['budget', 'revenue']:
    movies[f'log1p_{c}'] = movies[c].apply(lambda v: np.log1p(v) if pd.notna(v) and v > 0 else 0.0)

movies[['title', 'release_year', 'runtime', 'vote_average', 'vote_count', 'popularity', 'log1p_budget', 'log1p_revenue']].head(5)


,title,release_year,runtime,vote_average,vote_count,popularity,log1p_budget,log1p_revenue
0,Toy Story,1995.0,81.0,7.7,5415.0,21.946943,17.216708,19.738573
1,Jumanji,1995.0,104.0,6.9,2413.0,17.015539,17.989898,19.386893
2,Grumpier Old Men,1995.0,101.0,6.5,92.0,11.712900,0.000000,0.000000
3,Waiting to Exhale,1995.0,127.0,6.1,34.0,3.859495,16.588099,18.215526
4,Father of the Bride Part II,1995.0,106.0,5.7,173.0,8.387519,0.000000,18.153832


In [9]:
# =====================================================================
# A-6. 필요한 핵심 컬럼만 남긴 'movies_cleaned.csv' 저장
# =====================================================================
# Item-Based에서 사용할 가능성이 높은 feature들만 남겨서 저장한다.
movies_cleaned = movies[[
    'tmdbId',
    'title',
    'original_title',
    'original_language',
    'release_date',
    'release_year',
    'runtime',
    'vote_average',
    'vote_count',
    'popularity',
    'log1p_budget',
    'log1p_revenue',
    'genres_list',
    'main_genre',
]]

output_path_movies = OUTPUT_DIR / 'movies_cleaned.csv'
movies_cleaned.to_csv(output_path_movies, index=False)
print('movies_cleaned 저장 완료 →', output_path_movies.resolve())
print('movies_cleaned shape:', movies_cleaned.shape)
movies_cleaned.head(5)


movies_cleaned 저장 완료 → C:\Users\gram\Downloads\item_based-recommendation-system\processed\movies_cleaned.csv
movies_cleaned shape: (45435, 14)


,tmdbId,title,original_title,original_language,release_date,release_year,runtime,vote_average,vote_count,popularity,log1p_budget,log1p_revenue,genres_list,main_genre
0,862,Toy Story,Toy Story,en,1995-10-30,1995.0,81.0,7.7,5415.0,21.946943,17.216708,19.738573,"[Animation, Comedy, Family]",Animation
1,8844,Jumanji,Jumanji,en,1995-12-15,1995.0,104.0,6.9,2413.0,17.015539,17.989898,19.386893,"[Adventure, Fantasy, Family]",Adventure
2,15602,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,1995.0,101.0,6.5,92.0,11.712900,0.000000,0.000000,"[Romance, Comedy]",Romance
3,31357,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,1995.0,127.0,6.1,34.0,3.859495,16.588099,18.215526,"[Comedy, Drama, Romance]",Comedy
4,11862,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,1995.0,106.0,5.7,173.0,8.387519,0.000000,18.153832,[Comedy],Comedy


## B. `ratings` 정규화

Item-Based CF에서 **평점 기반 아이템-아이템 유사도** (예: cosine, Pearson)를 계산할 때,
사용자마다 평점을 주는 습관(항상 높게/낮게 주는 편향)을 제거하는 것이 중요하다.

따라서 `ratings.csv`에서 다음과 같은 전처리를 수행한다.

1. `timestamp` → datetime으로 변환 (시간 기반 분석/필터링 가능성 확보)
2. 사용자별 평균 평점을 기준으로 **mean-centering** 된 평점(`rating_centered`) 생성
   -  \( r'_{u,i} = r_{u,i} - \bar{r}_u \)
   - 이렇게 하면 user bias를 제거하고, 아이템 간의 패턴 비교가 더 안정적이 된다.
3. 필요하다면 최근 데이터만 사용하거나, 평점 수가 너무 적은 아이템/유저 필터링 가능


In [10]:
# =====================================================================
# B-1. ratings 로딩
# =====================================================================
ratings_raw = pd.read_csv(RATINGS_PATH)
print('원본 ratings shape:', ratings_raw.shape)
ratings_raw.head(3)


원본 ratings shape: (100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


In [11]:
# =====================================================================
# B-2. timestamp → datetime 변환
# =====================================================================
# UNIX timestamp(초 단위)를 실제 datetime으로 변환한다.
# 시간 기반 필터링(예: 최신 평점만 사용)이나, 평점 시계열 분석이 필요할 때 활용 가능하다.
ratings = ratings_raw.copy()
ratings['rating_datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings[['userId', 'movieId', 'rating', 'rating_datetime']].head(5)


,userId,movieId,rating,rating_datetime
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51


In [12]:
# =====================================================================
# B-3. 사용자별 mean-centering (rating_centered)
# =====================================================================
# Item-Based에서 아이템 간 유사도를 계산할 때,
# 각 사용자의 평점 수준 차이를 제거하기 위해 평균을 빼주는 것이 일반적이다.
#   r'_{u,i} = r_{u,i} - mean_rating_u
user_mean = ratings.groupby('userId')['rating'].mean()
ratings = ratings.join(user_mean, on='userId', rsuffix='_user_mean')

ratings['rating_centered'] = ratings['rating'] - ratings['rating_user_mean']

print('userId 1의 평균 평점:', user_mean.loc[1])
ratings[ratings['userId'] == 1].head(5)


userId 1의 평균 평점: 4.366379310344827


,userId,movieId,rating,timestamp,rating_datetime,rating_user_mean,rating_centered
0,1,1,4.0,964982703,2000-07-30 18:45:03,4.366379,-0.366379
1,1,3,4.0,964981247,2000-07-30 18:20:47,4.366379,-0.366379
2,1,6,4.0,964982224,2000-07-30 18:37:04,4.366379,-0.366379
3,1,47,5.0,964983815,2000-07-30 19:03:35,4.366379,0.633621
4,1,50,5.0,964982931,2000-07-30 18:48:51,4.366379,0.633621


In [13]:
# =====================================================================
# B-4. (선택) 너무 적게 평가된 영화/유저 필터링
# =====================================================================
# Item-Based 유사도 계산에서 극단적으로 평점 수가 적은 아이템/유저는
# 노이즈를 많이 유발할 수 있다.
# 예를 들어, 5개 미만의 평점만 가진 영화는 제외하는 식으로 필터링을 고려할 수 있다.
#
# 여기서는 기준만 예시로 보여주고, 실제 필터링은 주석 처리해둔다.
item_counts = ratings.groupby('movieId').size()
user_counts = ratings.groupby('userId').size()

print('평점 수 기준 예시:')
print('  영화별 평점 수 describe:', item_counts.describe())
print('  유저별 평점 수 describe:', user_counts.describe())

# 예시: 최소 평점 수 기준 (실제 기준은 실험을 통해 조정하는 것이 좋다)
MIN_ITEM_RATINGS = 5
MIN_USER_RATINGS = 5

# mask_item = ratings['movieId'].map(item_counts) >= MIN_ITEM_RATINGS
# mask_user = ratings['userId'].map(user_counts) >= MIN_USER_RATINGS
# ratings = ratings[mask_item & mask_user]

# print('필터링 후 ratings shape:', ratings.shape)


평점 수 기준 예시:
  영화별 평점 수 describe: count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
dtype: float64
  유저별 평점 수 describe: count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64


In [14]:
# =====================================================================
# B-5. 필요한 컬럼만 남긴 'ratings_cleaned.csv' 저장
# =====================================================================
# Item-Based 평점 유사도 계산에 필요한 핵심 컬럼만 저장한다.
ratings_cleaned = ratings[[
    'userId',
    'movieId',
    'rating',
    'rating_datetime',
    'rating_user_mean',
    'rating_centered',
    'timestamp',
]]

output_path_ratings = OUTPUT_DIR / 'ratings_cleaned.csv'
ratings_cleaned.to_csv(output_path_ratings, index=False)
print('ratings_cleaned 저장 완료 →', output_path_ratings.resolve())
print('ratings_cleaned shape:', ratings_cleaned.shape)
ratings_cleaned.head(5)


ratings_cleaned 저장 완료 → C:\Users\gram\Downloads\item_based-recommendation-system\processed\ratings_cleaned.csv
ratings_cleaned shape: (100836, 7)


,userId,movieId,rating,rating_datetime,rating_user_mean,rating_centered,timestamp
0,1,1,4.0,2000-07-30 18:45:03,4.366379,-0.366379,964982703
1,1,3,4.0,2000-07-30 18:20:47,4.366379,-0.366379,964981247
2,1,6,4.0,2000-07-30 18:37:04,4.366379,-0.366379,964982224
3,1,47,5.0,2000-07-30 19:03:35,4.366379,0.633621,964983815
4,1,50,5.0,2000-07-30 18:48:51,4.366379,0.633621,964982931


## C. `tags` 전처리

태그 데이터는 Item-Based에서 **콘텐츠 기반 유사도**를 계산할 때 중요한 텍스트 feature가 된다.
예를 들어, 태그들을 TF-IDF 벡터로 변환한 뒤, 영화 간 cosine similarity를 계산할 수 있다.

이를 위해 다음과 같은 전처리를 수행한다.

1. `timestamp` → datetime 변환 (시간 기반 필터링 가능)
2. 태그 문자열 전처리:
   - 소문자 변환
   - 앞뒤 공백 제거
   - 특수문자/불필요한 기호 제거
   - 공백 다중 → 단일 공백
3. 너무 짧거나 의미 없는 태그 제거 (예: 공백, 한 글자 등)
4. 영화별 태그를 하나의 문서처럼 합쳐서, 이후 TF-IDF 벡터화를 쉽게 할 수 있도록 준비


In [15]:
# =====================================================================
# C-1. tags 로딩
# =====================================================================
tags_raw = pd.read_csv(TAGS_PATH)
print('원본 tags shape:', tags_raw.shape)
tags_raw.head(5)


원본 tags shape: (3683, 4)


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [16]:
# =====================================================================
# C-2. timestamp → datetime 변환
# =====================================================================
tags = tags_raw.copy()
tags['tag_datetime'] = pd.to_datetime(tags['timestamp'], unit='s')
tags[['userId', 'movieId', 'tag', 'tag_datetime']].head(5)


,userId,movieId,tag,tag_datetime
0,2,60756,funny,2015-10-24 19:29:54
1,2,60756,Highly quotable,2015-10-24 19:29:56
2,2,60756,will ferrell,2015-10-24 19:29:52
3,2,89774,Boxing story,2015-10-24 19:33:27
4,2,89774,MMA,2015-10-24 19:33:20


In [17]:
# =====================================================================
# C-3. 태그 문자열 전처리 함수 정의
# =====================================================================
# - 영어 기준으로 단순한 규칙 기반 전처리를 수행한다.
# - 복잡한 lemmatization / stopword 제거는 향후 필요시 추가할 수 있다.
TAG_MIN_LEN = 2  # 너무 짧은 태그(1글자)는 노이즈일 가능성이 높으므로 제거 기준 예시

def normalize_tag(text: str) -> str:
    """
    태그 전처리 함수:
    1) 소문자 변환
    2) 양쪽 공백 제거
    3) 특수문자 제거 (알파벳/숫자/공백만 남김)
    4) 다중 공백 → 단일 공백
    """
    if not isinstance(text, str):
        return ''
    # 1) 소문자
    text = text.lower().strip()
    # 2) 특수문자 제거 (알파벳/숫자/공백만 허용)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # 3) 다중 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    return text

tags['tag_normalized'] = tags['tag'].apply(normalize_tag)

# 공백이거나 너무 짧은 태그 제거
mask_valid = tags['tag_normalized'].str.len() >= TAG_MIN_LEN
tags = tags[mask_valid]

print('전처리 후 tags shape:', tags.shape)
tags[['userId', 'movieId', 'tag', 'tag_normalized']].head(10)


전처리 후 tags shape: (3683, 6)


,userId,movieId,tag,tag_normalized
0,2,60756,funny,funny
1,2,60756,Highly quotable,highly quotable
2,2,60756,will ferrell,will ferrell
3,2,89774,Boxing story,boxing story
4,2,89774,MMA,mma
5,2,89774,Tom Hardy,tom hardy
6,2,106782,drugs,drugs
7,2,106782,Leonardo DiCaprio,leonardo dicaprio
8,2,106782,Martin Scorsese,martin scorsese
9,7,48516,way too long,way too long


In [18]:
# =====================================================================
# C-4. 영화별 태그를 하나의 문서처럼 합치기
# =====================================================================
# Item-Based에서 TF-IDF 기반 콘텐츠 유사도를 계산할 때,
# 영화 하나를 하나의 '문서(document)'로 보고,
# 그 영화에 달린 모든 태그를 공백으로 이어붙여서 사용한다.
movie_tag_corpus = (
    tags
    .groupby('movieId')['tag_normalized']
    .apply(lambda s: ' '.join(sorted(set(s))))  # 중복 태그는 제거(set), 정렬 후 연결
    .reset_index()
    .rename(columns={'tag_normalized': 'tags_joined'})
)

print('movie_tag_corpus shape:', movie_tag_corpus.shape)
movie_tag_corpus.head(10)


movie_tag_corpus shape: (1572, 2)


,movieId,tags_joined
0,1,fun pixar
1,2,fantasy game magic board game robin williams
2,3,moldy old
3,5,pregnancy remake
4,7,remake
5,11,politics president
6,14,politics president
7,16,mafia
8,17,jane austen
9,21,hollywood


In [19]:
# =====================================================================
# C-5. 태그 전처리 결과 저장
# =====================================================================
# 1) row 단위 태그 (각 사용자가 남긴 개별 태그 레코드)
output_path_tags_row = OUTPUT_DIR / 'tags_cleaned_row.csv'
tags_cleaned_row = tags[[
    'userId',
    'movieId',
    'tag',
    'tag_normalized',
    'tag_datetime',
    'timestamp',
]]
tags_cleaned_row.to_csv(output_path_tags_row, index=False)
print('tags_cleaned_row 저장 완료 →', output_path_tags_row.resolve())
print('tags_cleaned_row shape:', tags_cleaned_row.shape)

# 2) 영화별 태그 코퍼스 (한 영화당 하나의 태그 문자열)
output_path_tags_movie = OUTPUT_DIR / 'movie_tag_corpus.csv'
movie_tag_corpus.to_csv(output_path_tags_movie, index=False)
print('movie_tag_corpus 저장 완료 →', output_path_tags_movie.resolve())


tags_cleaned_row 저장 완료 → C:\Users\gram\Downloads\item_based-recommendation-system\processed\tags_cleaned_row.csv
tags_cleaned_row shape: (3683, 6)
movie_tag_corpus 저장 완료 → C:\Users\gram\Downloads\item_based-recommendation-system\processed\movie_tag_corpus.csv


## 마무리

이 노트북에서 수행한 전처리 결과로 다음 파일들이 생성된다.

- `processed/movies_cleaned.csv`
  - Item-Based에서 사용할 수 있는 영화 메타데이터 feature 정제본
- `processed/ratings_cleaned.csv`
  - 사용자별 mean-centering이 적용된 평점 데이터
- `processed/tags_cleaned_row.csv`
  - 전처리된 개별 태그 레코드
- `processed/movie_tag_corpus.csv`
  - 영화별로 합쳐진 태그 문자열 (TF-IDF 등 텍스트 벡터화 용도)

다음 단계(Item-Based 구현)에서는:
- `ratings_cleaned.csv`의 `rating_centered`를 사용하여 **평점 기반 아이템-아이템 유사도**를 계산하거나,
- `movies_cleaned.csv`와 `movie_tag_corpus.csv`를 사용하여 **콘텐츠 기반 아이템 유사도**를 계산하는 하이브리드 모델을 설계할 수 있다.
